# station_hour_bike_flow 기반 horizon-conditioned 재고 예측 End-to-End

이 노트북은 `현재 재고 + h시간 뒤 누적 순변화 예측` 구조를 위한 최종 작업 노트북이다.

- 데이터 로드
- `h시간 뒤 누적 순변화` 타깃 생성
- horizon-conditioned long dataset 생성
- baseline 회귀모델 학습
- horizon별 성능 평가
- `joblib` 모델 저장
- 입력 스키마와 메타데이터 저장
- Flutter / API 예시 산출

In [ ]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import joblib
except ImportError:
    joblib = None

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works' / '07_stock_prediction'
OUTPUT_DIR = WORK_DIR / '03_output_data'
FLOW_DIR = ROOT / '3조 공유폴더' / 'station_hour_bike_flow_2023_2025'

TRAIN_PATH = FLOW_DIR / 'station_hour_bike_flow_train_2023_2024.csv'
TEST_PATH = FLOW_DIR / 'station_hour_bike_flow_test_2025.csv'
TRAIN_HORIZONS = [1, 2, 3, 6]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR

## 1. 데이터 로드

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

for df in (train_df, test_df):
    df['time'] = pd.to_datetime(df['time'])

print(train_df.shape, test_df.shape)
train_df.head()

## 2. 데이터 정합성 점검

`bike_change = return_count - rental_count`가 성립하는지 확인한다.

In [ ]:
train_mismatch = ((train_df['return_count'] - train_df['rental_count']) != train_df['bike_change']).sum()
test_mismatch = ((test_df['return_count'] - test_df['rental_count']) != test_df['bike_change']).sum()
print({'train_mismatch': int(train_mismatch), 'test_mismatch': int(test_mismatch)})

## 3. 기본 피처 생성

현재 상태와 최근 이력을 이용해 `h시간 뒤 누적 순변화`를 설명하는 기본 피처를 만든다.

In [ ]:
def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['station_id', 'time']).copy()
    group = df.groupby('station_id', sort=False)

    for col in ['rental_count', 'return_count', 'bike_change', 'bike_count_index']:
        df[f'{col}_lag_1'] = group[col].shift(1)
        df[f'{col}_lag_2'] = group[col].shift(2)

    shifted_bike_change = group['bike_change'].shift(1)
    df['bike_change_rollmean_3'] = shifted_bike_change.rolling(3).mean().reset_index(level=0, drop=True)
    df['bike_change_rollstd_3'] = shifted_bike_change.rolling(3).std().reset_index(level=0, drop=True)
    df['bike_change_rollmean_6'] = shifted_bike_change.rolling(6).mean().reset_index(level=0, drop=True)
    df['bike_change_rollstd_6'] = shifted_bike_change.rolling(6).std().reset_index(level=0, drop=True)
    df['bike_change_rollmean_24'] = shifted_bike_change.rolling(24).mean().reset_index(level=0, drop=True)
    df['bike_change_rollstd_24'] = shifted_bike_change.rolling(24).std().reset_index(level=0, drop=True)

    df['is_weekend'] = (df['weekday'] >= 5).astype(int)
    df['is_commute_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df['is_lunch_hour'] = df['hour'].isin([11, 12, 13]).astype(int)
    df['is_night_hour'] = df['hour'].isin([22, 23, 0, 1, 2, 3, 4]).astype(int)
    return df

train_df = add_base_features(train_df)
test_df = add_base_features(test_df)
train_df.head()

## 4. horizon-conditioned 타깃 생성

고정된 `2시간 후` 모델이 아니라, `horizon_hours`를 입력으로 받는 단일 회귀식을 만든다.

- `target_net_change_h = bike_count_index(t+h) - bike_count_index(t)`
- 같은 의미로 `sum(bike_change(t+1) ... bike_change(t+h))`
- 학습 데이터는 `(station_id, time, horizon_hours)` 단위의 long-format으로 만든다.

In [ ]:
def build_horizon_dataset(df: pd.DataFrame, horizons: list[int]) -> pd.DataFrame:
    base = df.sort_values(['station_id', 'time']).copy()
    group = base.groupby('station_id', sort=False)
    frames = []

    for h in horizons:
        horizon_df = base.copy()
        horizon_df['horizon_hours'] = h
        horizon_df['horizon_hours_sq'] = h ** 2
        horizon_df['target_net_change_h'] = group['bike_count_index'].shift(-h) - base['bike_count_index']
        horizon_df['target_bike_count_index_h'] = group['bike_count_index'].shift(-h)
        frames.append(horizon_df)

    result = pd.concat(frames, ignore_index=True)
    result = result.dropna(subset=['target_net_change_h']).copy()
    return result

train_long = build_horizon_dataset(train_df, TRAIN_HORIZONS)
test_long = build_horizon_dataset(test_df, TRAIN_HORIZONS)

print(train_long.shape, test_long.shape)
train_long[['station_id', 'time', 'horizon_hours', 'bike_count_index', 'target_net_change_h']].head(10)

## 5. 학습용 컬럼 정의

In [ ]:
feature_cols = [
    'station_id', 'hour', 'weekday', 'month',
    'horizon_hours', 'horizon_hours_sq',
    'is_weekend', 'is_commute_hour', 'is_lunch_hour', 'is_night_hour',
    'rental_count', 'return_count', 'bike_change', 'bike_count_index',
    'rental_count_lag_1', 'rental_count_lag_2',
    'return_count_lag_1', 'return_count_lag_2',
    'bike_change_lag_1', 'bike_change_lag_2',
    'bike_count_index_lag_1', 'bike_count_index_lag_2',
    'bike_change_rollmean_3', 'bike_change_rollstd_3',
    'bike_change_rollmean_6', 'bike_change_rollstd_6',
    'bike_change_rollmean_24', 'bike_change_rollstd_24',
]

categorical_cols = ['station_id']
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

train_model = train_long.copy()
test_model = test_long.copy()
train_model.shape, test_model.shape

## 6. baseline 회귀모델 학습

1차 baseline은 Ridge로 두고, `horizon_hours`를 함께 입력한다.

In [ ]:
def build_model(alpha: float = 2.0) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]), numeric_cols),
        ]
    )
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', Ridge(alpha=2.0)),
    ])

def evaluate(y_true, pred):
    return {
        'rmse': float(mean_squared_error(y_true, pred) ** 0.5),
        'mae': float(mean_absolute_error(y_true, pred)),
        'r2': float(r2_score(y_true, pred)),
    }

model = build_model(alpha=2.0)
model.fit(train_model[feature_cols], train_model['target_net_change_h'])

pred = model.predict(test_model[feature_cols])
overall_metrics = evaluate(test_model['target_net_change_h'], pred)
overall_metrics

## 7. horizon별 성능 점검

In [ ]:
pred_df = pd.DataFrame({
    'station_id': test_model['station_id'].values,
    'time': test_model['time'].values,
    'horizon_hours': test_model['horizon_hours'].values,
    'actual_net_change': test_model['target_net_change_h'].values,
    'predicted_net_change': pred,
})

metrics_by_horizon = []
for h, group_df in pred_df.groupby('horizon_hours'):
    row = {'horizon_hours': int(h), **evaluate(group_df['actual_net_change'], group_df['predicted_net_change'])}
    metrics_by_horizon.append(row)

metrics_by_horizon = pd.DataFrame(metrics_by_horizon).sort_values('horizon_hours').reset_index(drop=True)
metrics_by_horizon

## 8. 산출물 저장

In [ ]:
metrics_by_horizon.to_csv(OUTPUT_DIR / 'metrics_by_horizon.csv', index=False)
pred_df.to_csv(OUTPUT_DIR / 'predictions_by_horizon.csv', index=False)
train_model.to_csv(OUTPUT_DIR / 'stock_horizon_train.csv', index=False)

feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coef_df = pd.DataFrame({
    'feature_name': feature_names,
    'coefficient': model.named_steps['model'].coef_,
})
coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coefficient', ascending=False)
coef_df.to_csv(OUTPUT_DIR / 'feature_importance_horizon.csv', index=False)

schema = {
    'feature_cols': feature_cols,
    'categorical_cols': categorical_cols,
    'numeric_cols': numeric_cols,
    'target': 'target_net_change_h',
    'target_definition': 'bike_count_index(t+h) - bike_count_index(t)',
    'runtime_required_fields': ['station_id', 'request_time', 'horizon_hours', 'current_stock'],
    'notes': {
        'current_stock': '실시간 API 입력이며, 학습 데이터에는 포함되지 않는다.',
        'horizon_hours': '수식에 직접 들어가는 미래 시간 입력값',
    },
}
with open(OUTPUT_DIR / 'model_input_schema.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

metadata = {
    'model_name': 'stock_model_horizon',
    'train_horizons': TRAIN_HORIZONS,
    'train_rows': int(len(train_model)),
    'test_rows': int(len(test_model)),
    'overall_metrics': overall_metrics,
    'metrics_by_horizon': metrics_by_horizon.to_dict(orient='records'),
    'data_source': str(FLOW_DIR),
}
with open(OUTPUT_DIR / 'model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

if joblib is not None:
    joblib.dump(model, OUTPUT_DIR / 'stock_model_horizon.joblib')

metadata

## 9. 운영 계산 예시

실시간 API에서 `current_stock`를 받으면 아래처럼 `h시간 뒤 재고`를 계산한다.

In [ ]:
example_row = test_model.iloc[0][feature_cols].copy()
example_current_stock = 12
example_row['horizon_hours'] = 6
example_row['horizon_hours_sq'] = 36

example_pred_net_change_h = float(model.predict(pd.DataFrame([example_row]))[0])
stock_t_plus_h = example_current_stock + example_pred_net_change_h

sample_request = {
    'station_id': int(example_row['station_id']),
    'request_time': str(test_model.iloc[0]['time']),
    'horizon_hours': int(example_row['horizon_hours']),
    'current_stock': example_current_stock,
}
sample_response = {
    'predicted_net_change_h': example_pred_net_change_h,
    'predicted_stock_h': stock_t_plus_h,
}

with open(OUTPUT_DIR / 'sample_inference_request.json', 'w', encoding='utf-8') as f:
    json.dump(sample_request, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / 'sample_inference_response.json', 'w', encoding='utf-8') as f:
    json.dump(sample_response, f, ensure_ascii=False, indent=2)

{
    'request': sample_request,
    'response': sample_response,
}